# Práctica: clasificando reseñas de compradores

En `reseñas.csv` tienes 20 reseñas de un e-commerce, cada una con su etiqueta de sentimiento (`positiva` o `negativa`). Es un dataset chico a propósito: no se trata de lograr el mejor accuracy posible, sino de que construyas el pipeline completo con tus propias manos y entiendas qué hace cada paso.
 
Antes de escribir código, leé las 20 reseñas del CSV a mano. Fijate qué palabras aparecen repetidas en las positivas y cuáles en las negativas. Esa intuición te va a servir después para saber si el modelo está aprendiendo algo razonable o si está fallando por algo que vos ya podías anticipar.

## Ejercicio 1: cargar y explorar
 
Cargá el CSV con Pandas. Contá cuántas reseñas hay de cada clase (`positiva` / `negativa`). Esto no es un paso decorativo: si las clases estuvieran muy desbalanceadas (por ejemplo 18 positivas y 2 negativas), un modelo podría lograr un accuracy alto simplemente prediciendo siempre "positiva", sin haber aprendido nada útil. Antes de entrenar cualquier modelo de clasificación, siempre chequeá el balance de clases.

In [15]:
import pandas as pd
df_reseñas = pd.read_csv("reseñas.csv", sep=";")


# df_reseñas.head()
# df_reseñas.columns
# df_reseñas['texto,sentimiento'].value_counts()

# Contar las clases dentro de la columna específica
conteo_clases = df_reseñas['texto,sentimiento'].value_counts()
print(conteo_clases)

# conteo = df_reseñas['texto,sentimiento'].value_counts()
# print(conteo)

texto,sentimiento
el producto llegó en perfecto estado y antes de lo esperado,positiva       1
pésima experiencia, el paquete llegó abierto y faltaban piezas,negativa    1
muy buena relación precio calidad, lo volvería a comprar,positiva          1
el vendedor nunca respondió mis mensajes, mala atención,negativa           1
superó mis expectativas, calidad excelente,positiva                        1
se rompió a la semana de uso, no lo recomiendo,negativa                    1
envío rapidísimo, tal cual la descripción,positiva                         1
la caja llegó totalmente destruida y el producto dañado,negativa           1
funciona perfecto, instalación sencilla y buen material,positiva           1
tardó un mes en llegar y encima vino incompleto,negativa                   1
excelente atención del vendedor, resolvió todas mis dudas,positiva         1
la calidad es muy inferior a lo que mostraban las fotos,negativa           1
quedé conforme, cumplió con lo prometido,positiva         

## Ejercicio 2: tokenizar sin librerías
 
Escribí una función `tokenizar_simple(texto)` que reciba un string y devuelva una lista de palabras en minúscula, sin signos de puntuación, usando solo métodos nativos de Python (sin NLTK ni spaCy). Pista: puede que necesites la librería `string` y su lista de puntuación, además de `.split()`.
 
Después, aplicá `word_tokenize` de NLTK sobre la misma reseña y compará los resultados. ¿En qué casos tu función simple se equivoca o produce un resultado distinto al de NLTK? Escribí al menos un ejemplo concreto de una reseña del dataset donde la diferencia se note.

In [16]:
%pip install nltk

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Giovanni\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [18]:
import string
import nltk
from nltk.tokenize import word_tokenize

# Descargamos el paquete de puntuación de NLTK (solo hace falta una vez)
nltk.download('punkt')
# Nota: Si te da un error pidiendo 'punkt_tab', cambia la línea de arriba por nltk.download('punkt_tab')

def tokenizar_simple(texto):
    texto = texto.lower()
    # Sumamos ¡ y ¿ que no vienen en el string.punctuation por defecto
    signos_a_eliminar = string.punctuation + '¡¿'
    tabla_limpieza = str.maketrans('', '', signos_a_eliminar)
    texto_limpio = texto.translate(tabla_limpieza)
    return texto_limpio.split()

# Tu ejemplo para comparar
reseña_ejemplo = "¡Terrible experiencia! El Wi-Fi del hotel era malísimo... y el desayuno costaba $10.50 extra."

tokens_simples = tokenizar_simple(reseña_ejemplo)
tokens_nltk = word_tokenize(reseña_ejemplo, language='spanish')

print("=== FUNCIÓN SIMPLE ===")
print(tokens_simples)
print("\n=== NLTK ===")
print(tokens_nltk)

=== FUNCIÓN SIMPLE ===
['terrible', 'experiencia', 'el', 'wifi', 'del', 'hotel', 'era', 'malísimo', 'y', 'el', 'desayuno', 'costaba', '1050', 'extra']

=== NLTK ===
['¡Terrible', 'experiencia', '!', 'El', 'Wi-Fi', 'del', 'hotel', 'era', 'malísimo', '...', 'y', 'el', 'desayuno', 'costaba', '$', '10.50', 'extra', '.']


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Giovanni\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


## Ejercicio 3: el efecto de las stopwords
 
Tomá la reseña `"no lo recomiendo, una pérdida de dinero total"` (o equivalente del dataset) y sacale las stopwords en español con NLTK. Mirá el resultado.
 
Ahora respondé sin correr más código, solo pensando: si esta reseña fuera parte de un modelo de Bag of Words que sacó stopwords, y la palabra "no" desapareció, ¿qué información se perdió? Buscá en el dataset si hay alguna otra reseña donde sacar "no" cambiaría el sentido de la frase. Este ejercicio no tiene una única respuesta "correcta" en código: el objetivo es que argumentes cuándo sacar stopwords ayuda y cuándo perjudica.

In [19]:
Esto perjudica si el Data Set se toma en cuenta los sentimientos de las reseñas, ya que la función simple no reconoce los signos de puntuación y los elimina, mientras que NLTK los mantiene como tokens separados. Esto puede afectar el análisis de sentimientos, ya que los signos de puntuación pueden influir en la interpretación del texto. Por ejemplo, un signo de exclamación puede indicar una emoción fuerte, mientras que su ausencia podría cambiar el tono de la reseña.

SyntaxError: invalid syntax (3284801159.py, line 1)

## Ejercicio 4: vectorizar con TF-IDF
 
Usando `TfidfVectorizer` de sklearn, vectorizá las 20 reseñas del dataset completo (no hace falta separar en train/test todavía). Imprimí el vocabulario completo que generó el vectorizador con `.get_feature_names_out()`.
 
Buscá en ese vocabulario las 5 palabras con el IDF más alto (es decir, las más "raras" o distintivas del corpus) y las 5 con el IDF más bajo. Podés acceder a esos valores con el atributo `.idf_` del vectorizador, en el mismo orden que el vocabulario. ¿Las palabras con IDF alto te parecen informativas sobre el sentimiento de la reseña donde aparecen?

In [ ]:
%pip install scikit-learn

   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   ----- ---------------------------------- 1.0/8.3 MB 5.3 MB/s eta 0:00:02
   ---------- ----------------------------- 2.1/8.3 MB 5.2 MB/s eta 0:00:02
   ---------------- ----------------------- 3.4/8.3 MB 5.4 MB/s eta 0:00:01
   --------------------- ------------------ 4.5/8.3 MB 5.5 MB/s eta 0:00:01
   --------------------------- ------------ 5.8/8.3 MB 5.5 MB/s eta 0:00:01
   -------------------------------- ------- 6.8/8.3 MB 5.5 MB/s eta 0:00:01
   ------------------------------------- -- 7.9/8.3 MB 5.5 MB/s eta 0:00:01
   ---------------------------------------- 8.3/8.3 MB 5.3 MB/s  0:00:01
   ---------------------------------------- 0.0/37.3 MB ? eta -:--:--
    --------------------------------------- 0.8/37.3 MB 5.2 MB/s eta 0:00:07
   -- ------------------------------------- 2.1/37.3 MB 5.4 MB/s eta 0:00:07
   --- ------------------------------------ 3.4/37.3 MB 5.4 MB/s eta 0:00:07
   ---- -----------------


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Cargamos los datos forzando la limpieza de caracteres invisibles y espacios
df_reseñas = pd.read_csv("reseñas.csv", sep=",", encoding='utf-8-sig')
df_reseñas.columns = df_reseñas.columns.str.strip() 

# 2. Inicializamos el vectorizador
vectorizador = TfidfVectorizer()

# 3. Ajustamos el vectorizador a nuestra columna de textos y transformamos
# ¡Ahora Pandas encontrará 'texto' sin problema!
matriz_tfidf = vectorizador.fit_transform(df_reseñas['texto'])

# 4. Extraemos el vocabulario y los puntajes IDF
vocabulario = vectorizador.get_feature_names_out()
valores_idf = vectorizador.idf_

# 5. Creamos un DataFrame auxiliar para ver todo ordenado
df_idf = pd.DataFrame({
    'Palabra': vocabulario, 
    'IDF': valores_idf
})

# Ordenamos el DataFrame de menor a mayor IDF
df_idf_ordenado = df_idf.sort_values(by='IDF')

print("=== 5 PALABRAS CON IDF MÁS BAJO (Comunes) ===")
print(df_idf_ordenado.head(5))
print("\n" + "="*40 + "\n")

print("=== 5 PALABRAS CON IDF MÁS ALTO (Raras/Informativas) ===")
print(df_idf_ordenado.tail(5))

=== 5 PALABRAS CON IDF MÁS BAJO (Comunes) ===
   Palabra       IDF
28      el  2.098612
49      la  2.098612
52   llegó  2.252763
53      lo  2.252763
22      de  2.435085


=== 5 PALABRAS CON IDF MÁS ALTO (Raras/Informativas) ===
      Palabra       IDF
99        una  3.351375
100       uso  3.351375
102      vino  3.351375
103  volvería  3.351375
104        ya  3.351375
